In [1]:
!pip install catboost

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error as mse
from sklearn.model_selection import KFold, StratifiedKFold, GroupKFold
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import os
from sklearn.preprocessing import MinMaxScaler
import warnings
import ast

warnings.simplefilter('ignore')

def rmse(y_t, y_p):
    return np.sqrt(mse(y_t, y_p))

In [3]:
DATA_PATH = '.'
train = pd.read_csv(DATA_PATH + os.sep + 'Train.csv')
test = pd.read_csv(DATA_PATH + os.sep + 'Test.csv')
ss = pd.read_csv(DATA_PATH + os.sep + 'SampleSubmission.csv')

train['is_train'] = 1
test['is_train'] = 0

ID_COL, TARGET_COLS = 'ID', ['m1', 'm2', 'm3', 'm4', 'm5', 'm6']

In [4]:
meta = pd.read_csv('metadata.csv')

In [5]:
df = train.append(test).reset_index(drop=True)

In [6]:
df['month'] = df['TransactionDates'].apply(lambda x: [int(e.split('-')[0]) for e in ast.literal_eval(x)])
df['year'] = df['TransactionDates'].apply(lambda x: [int(e.split('-')[1]) for e in ast.literal_eval(x)])
df['PaymentsHistory'] = df['PaymentsHistory'].apply(lambda x: [float(e) for e in ast.literal_eval(x)])

def get_month_year(x):
  return [y*12 + m for y, m in zip(x['year'], x['month'])]

df['month_year'] = df[['month', 'year']].apply(lambda x: get_month_year(x), axis=1)

def get_cons_len_seq(x):
  return  [-1] * (41 - len(x)) + x

df['month'] = df['month'].apply(lambda x: get_cons_len_seq(x))
df['year'] = df['year'].apply(lambda x: get_cons_len_seq(x))
df['PaymentsHistory'] = df['PaymentsHistory'].apply(lambda x: get_cons_len_seq(x))
df['month_year'] = df['month_year'].apply(lambda x: get_cons_len_seq(x))

In [7]:
df['order'] = df['month_year'].apply(lambda x: np.argsort(x))
for c in ['month', 'year', 'PaymentsHistory', 'month_year']:

  df[c] = df[[c, 'order']].apply(lambda x: list(np.array(x[c])[x['order']]), axis=1)

del df['order']

In [8]:
def get_total_months_spent(x):
  res = []
  min_val = np.min([e for e in x if e != -1])


  for e in x:
    if e == -1:
      res.append(-1)
    else:
      res.append(e - min_val)

  return res

df['total_months_spent_paying'] = df['month_year'].apply(lambda x: get_total_months_spent(x))


def get_paid_till_now(x):
  i = 0
  res = []
  curr_sum = 0
  for e in x:
    if e == -1:
      res.append(e)
    else:
      i += 1
      curr_sum += e
      res.append(curr_sum)
  return res

def get_remaining_till_now(x, total):
  i = 0
  res = []
  curr_sum = 0
  for e in x:
    if e == -1:
      res.append(-1)
    else:
      i += 1
      curr_sum += e
      res.append(total - curr_sum)
  return res


df['paid_till_now'] = df['PaymentsHistory'].apply(lambda x: get_paid_till_now(x))
df['TotalContractValue'] = df['ID'].map(meta.set_index('ID')['TotalContractValue'])
df['remaining_till_now'] = df[['PaymentsHistory', 'TotalContractValue']].\
apply(lambda x: get_remaining_till_now(x['PaymentsHistory'], x['TotalContractValue']), axis=1)

In [9]:
cols_dct = {}
df_new = df[[ID_COL, 'is_train'] + TARGET_COLS].copy()

for c in ['month', 'year', 'PaymentsHistory', 'month_year', 'paid_till_now', 'total_months_spent_paying', 'remaining_till_now']:
  ser = df[c]
  tmp_df = pd.DataFrame.from_dict(dict(zip(ser.index, ser.values))).T
  tmp_df.columns = [f'{c}_{i}' for i in tmp_df.columns]
  df_new = pd.concat([df_new, tmp_df], axis=1)
  cols_dct[c] = tmp_df.columns

In [10]:
id_vars = [c for c in df_new.columns if c not in TARGET_COLS]
df_melted = pd.melt(df_new, id_vars=id_vars, value_vars=TARGET_COLS, var_name='target_month', value_name='target_value')
df_melted['target_month_int'] = df_melted['target_month'].apply(lambda x: int(x[1]))

In [11]:
df_melted['total_payments'] = df_melted[cols_dct['month']].apply(lambda x: 41 - sum(x == -1), axis=1)

In [12]:
df_melted['total_paid'] = df_melted[cols_dct['PaymentsHistory']].apply(lambda x: sum([e if e != -1 else 0 for e in x]), axis=1)
df_melted['max_paid'] = df_melted[cols_dct['PaymentsHistory']].apply(lambda x: max([e for e in x if e != -1]), axis=1)
df_melted['min_paid'] = df_melted[cols_dct['PaymentsHistory']].apply(lambda x: min([e for e in x if e != -1]), axis=1)
df_melted['mean_paid'] = df_melted[cols_dct['PaymentsHistory']].apply(lambda x: np.mean([e for e in x if e != -1]), axis=1)
df_melted['range_paid'] = df_melted[cols_dct['PaymentsHistory']].apply(lambda x: np.ptp([e for e in x if e != -1]), axis=1)
df_melted['std_paid'] = df_melted[cols_dct['PaymentsHistory']].apply(lambda x: np.std([e for e in x if e != -1]), axis=1)
df_melted['first_paid'] = df_melted[cols_dct['PaymentsHistory']].apply(lambda x: ([e for e in x if e != -1][0]), axis=1)
df_melted['last_paid'] = df_melted[cols_dct['PaymentsHistory']].apply(lambda x: ([e for e in x if e != -1][-1]), axis=1)
df_melted['total_months_paid'] = df_melted[cols_dct['month_year']].apply(lambda x: np.ptp([e for e in x if e != -1]), axis=1)
df_melted['min_month_year_paid'] = df_melted[cols_dct['month_year']].apply(lambda x: np.max([e for e in x if e != -1]), axis=1)
df_melted['max_month_year_paid'] = df_melted[cols_dct['month_year']].apply(lambda x: np.min([e for e in x if e != -1]), axis=1)
df_melted['median_month_year_paid'] = df_melted[cols_dct['month_year']].apply(lambda x: np.median([e for e in x if e != -1]), axis=1)

In [13]:
df_melted['total_paid_/_total_months_paid'] = df_melted['total_paid']/df_melted['total_months_paid']

In [14]:
df_melted['total_paid_mean_per_total_payments'] = df_melted.groupby(['total_payments'])['total_paid'].transform('mean')
df_melted['total_paid_mean_per_min_month_year_paid'] = df_melted.groupby(['min_month_year_paid'])['total_paid'].transform('mean')

In [15]:
def get_monthly(x, i):
  try:
    return [e for e in x if e != -1][i]
  except:
    return -1

def get_monthly_diff(x, i):
  try:
    tmp = [e for e in x if e != -1]
    return tmp[i] - tmp[i+1]
  except:
    return -1

for c in ['remaining_till_now', 'total_months_spent_paying', 'paid_till_now', 'PaymentsHistory']:
  for i in range(41):
    df_melted[f'{c}_{i}_payment_no'] = df_melted[cols_dct['remaining_till_now']].apply(lambda x: get_monthly(x, i), axis=1)

  for i in range(-1, -41, -1):
    df_melted[f'{c}_{i}_payment_no_from_last'] = df_melted[cols_dct['remaining_till_now']].apply(lambda x: get_monthly(x, i), axis=1)

In [16]:
for c in ['remaining_till_now', 'total_months_spent_paying', 'paid_till_now', 'PaymentsHistory']:
  for i in range(41):
    df_melted[f'{c}_{i}_payment_no_diff_to_next'] = df_melted[cols_dct['remaining_till_now']].apply(lambda x: get_monthly_diff(x, i), axis=1)

In [17]:
def get_diff_to_first_paid(x):
  seq = [e for e in x if e != -1]
  return seq[1] - seq[0]

def get_diff_to_last_paid(x):
  seq = [e for e in x if e != -1]
  return seq[-1] - seq[-2]

df_melted['diff_month_year_last_paid'] = df_melted[cols_dct['month_year']].apply(lambda x: get_diff_to_first_paid(x), axis=1)
df_melted['diff_month_year_first_paid'] = df_melted[cols_dct['month_year']].apply(lambda x: get_diff_to_last_paid(x), axis=1)

df_melted['diff_PaymentsHistory_last_paid'] = df_melted[cols_dct['PaymentsHistory']].apply(lambda x: get_diff_to_first_paid(x), axis=1)
df_melted['diff_PaymentsHistory_first_paid'] = df_melted[cols_dct['PaymentsHistory']].apply(lambda x: get_diff_to_last_paid(x), axis=1)

df_melted['min_month_year_paid_freq'] = df_melted['min_month_year_paid'].map(df_melted['min_month_year_paid'].value_counts())
df_melted['max_month_year_paid_freq'] = df_melted['max_month_year_paid'].map(df_melted['max_month_year_paid'].value_counts())

In [18]:
cat_cols = ['PaymentMethod', 'rateTypeEntity', 'MainApplicantGender', 'Region', 'Town','Occupation','SupplierName']

meta[cat_cols] = meta[cat_cols].apply(lambda x: pd.factorize(x)[0])

In [19]:
not_use_meta_cols = [c for c in meta.columns if 'Date' in c]

In [20]:
meta_datetime_cols = ['RegistrationDate', 'ExpectedTermDate', 'FirstPaymentDate', 'LastPaymentDate']
for c in meta_datetime_cols:
  meta[c] = pd.to_datetime(meta[c])
  meta[f'{c}_days_since_start'] = (meta[c] - meta[c].min()).dt.days


for i in range(len(meta_datetime_cols)):
  for j in range(i+1, len(meta_datetime_cols)):
    name = f'{meta_datetime_cols[i]} - {meta_datetime_cols[j]}'
    meta[name] = (meta[meta_datetime_cols[i]] - meta[meta_datetime_cols[j]]).dt.days

In [21]:
use_meta_cols = [c for c in meta.columns if c not in not_use_meta_cols]

In [22]:
df_melted = pd.merge(df_melted, meta[use_meta_cols], on = "ID", how='left')

In [23]:
df_melted['remaining_payment2'] = df_melted['TotalContractValue'] - df_melted['total_paid']
df_melted['remaining_payment'] = df_melted['TotalContractValue'] - df_melted['Deposit'] 

In [24]:
df_melted['f1'] = df_melted['TotalContractValue'] / (-df_melted['FirstPaymentDate - LastPaymentDate'])
df_melted['f2'] = df_melted['remaining_payment2'] / (-df_melted['FirstPaymentDate - LastPaymentDate'])

In [25]:
cat_cols = ['PaymentMethod', 'rateTypeEntity', 'MainApplicantGender', 'Region', 'Town','Occupation','SupplierName']
num_cols = ['total_paid']

for c in cat_cols:
  for n in num_cols:
    df_melted[c + '_' + n + '_mean'] = df_melted.groupby(c)[n].transform('mean')

In [26]:
for c in ['PaymentsHistory', 'month_year']:
  cols = cols_dct[c]

  for i in range(1, len(cols) - 2):
    name = f'{cols[i]} - {cols[i+1]}'
    df_melted[name] = df_melted[cols[i]] - df_melted[cols[i+1]]

In [27]:
drop_cols = [c for c in df_melted.columns if sum(df_melted[c] == -1) > 17000]
len(drop_cols)

152

In [28]:
train, test = df_melted[df_melted['is_train'] == 1].reset_index(drop=True), df_melted[df_melted['is_train'] == 0].reset_index(drop=True)

In [29]:
TARGET_COL = 'target_value'
METRIC_NAME = 'rmse'

In [30]:
fts = [c for c in train.columns if c not in [ID_COL, TARGET_COL, 'created_at', 'is_train', 'target_month'] + drop_cols]

In [31]:
N_FOLDS = 10
folds = GroupKFold(N_FOLDS)
oofs, preds = np.zeros(len(train)), np.zeros(len(test))
fi_df = pd.DataFrame()

for f, (trn_idx, val_idx) in enumerate(folds.split(train, train[TARGET_COL], train[ID_COL])):

    X_trn, y_trn = train.iloc[trn_idx][fts], train[TARGET_COL].iloc[trn_idx]
    X_val, y_val = train.iloc[val_idx][fts], train[TARGET_COL].iloc[val_idx]

    clf = LGBMRegressor(n_estimators=10000, learning_rate = 0.05, colsample_bytree=0.3, reg_lambda=1, reg_alpha=2, importance_type='gain')

    clf.fit(X_trn, y_trn, eval_set=[(X_val, y_val)], verbose=0, early_stopping_rounds=150)

    vp = clf.predict(X_val)
    tp = clf.predict(test[fts])

    score = round(rmse(y_val, vp), 3)

    print(f'Fold: {f} {METRIC_NAME}: {score}')

    oofs[val_idx] = vp
    preds += tp/N_FOLDS

    fi_df = fi_df.append(pd.DataFrame({'ft': fts, 'imp': clf.feature_importances_}))

oof_score = round(rmse(train[TARGET_COL], oofs), 3)
print(f'\nOOF {METRIC_NAME}: {oof_score}')

Fold: 0 rmse: 908.194
Fold: 1 rmse: 753.468
Fold: 2 rmse: 695.752
Fold: 3 rmse: 777.614
Fold: 4 rmse: 741.457
Fold: 5 rmse: 1164.052
Fold: 6 rmse: 702.312
Fold: 7 rmse: 766.677
Fold: 8 rmse: 751.958
Fold: 9 rmse: 725.174

OOF rmse: 809.84


In [32]:
fi = fi_df.groupby('ft')['imp'].mean().sort_values(ascending=False)

len(fi), sum(fi != 0)

(757, 720)

In [33]:
fts = fi[:100].index.tolist()

In [34]:
len(fts)

100

In [35]:
N_FOLDS = 25
folds = GroupKFold(N_FOLDS)
oofs, preds = np.zeros(len(train)), np.zeros(len(test))
fi_df = pd.DataFrame()

for f, (trn_idx, val_idx) in enumerate(folds.split(train, train[TARGET_COL], train[ID_COL])):

    X_trn, y_trn = train.iloc[trn_idx][fts], train[TARGET_COL].iloc[trn_idx]
    X_val, y_val = train.iloc[val_idx][fts], train[TARGET_COL].iloc[val_idx]

    clf = LGBMRegressor(n_estimators=10000, learning_rate = 0.05, colsample_bytree=0.3, reg_lambda=1, reg_alpha=2, importance_type='gain')

    clf.fit(X_trn, y_trn, eval_set=[(X_val, y_val)], verbose=0, early_stopping_rounds=150)

    vp = clf.predict(X_val)
    tp = clf.predict(test[fts])

    score = round(rmse(y_val, vp), 3)

    print(f'Fold: {f} {METRIC_NAME}: {score}')

    oofs[val_idx] = vp
    preds += tp/N_FOLDS

    fi_df = fi_df.append(pd.DataFrame({'ft': fts, 'imp': clf.feature_importances_}))

oof_score = round(rmse(train[TARGET_COL], oofs), 3)
print(f'\nOOF {METRIC_NAME}: {oof_score}')

lgb_oofs = oofs.copy()
lgb_preds = preds.copy()

Fold: 0 rmse: 993.188
Fold: 1 rmse: 721.728
Fold: 2 rmse: 707.607
Fold: 3 rmse: 851.195
Fold: 4 rmse: 715.574
Fold: 5 rmse: 890.246
Fold: 6 rmse: 729.374
Fold: 7 rmse: 720.929
Fold: 8 rmse: 716.851
Fold: 9 rmse: 655.134
Fold: 10 rmse: 985.705
Fold: 11 rmse: 753.503
Fold: 12 rmse: 692.896
Fold: 13 rmse: 819.041
Fold: 14 rmse: 747.146
Fold: 15 rmse: 1389.043
Fold: 16 rmse: 715.597
Fold: 17 rmse: 741.777
Fold: 18 rmse: 699.712
Fold: 19 rmse: 690.698
Fold: 20 rmse: 873.742
Fold: 21 rmse: 683.93
Fold: 22 rmse: 743.671
Fold: 23 rmse: 681.428
Fold: 24 rmse: 821.583

OOF rmse: 804.051


In [36]:
N_FOLDS = 25
folds = GroupKFold(N_FOLDS)
oofs, preds = np.zeros(len(train)), np.zeros(len(test))
fi_df = pd.DataFrame()

for f, (trn_idx, val_idx) in enumerate(folds.split(train, train[TARGET_COL], train[ID_COL])):

    X_trn, y_trn = train.iloc[trn_idx][fts], train[TARGET_COL].iloc[trn_idx]
    X_val, y_val = train.iloc[val_idx][fts], train[TARGET_COL].iloc[val_idx]

    clf = CatBoostRegressor(n_estimators=10000, learning_rate = 0.05, rsm=0.3, max_depth=4,
                            random_state=2)

    clf.fit(X_trn, y_trn, eval_set=[(X_val, y_val)], verbose=0, early_stopping_rounds=150)

    vp = clf.predict(X_val)
    tp = clf.predict(test[fts])

    score = round(rmse(y_val, vp), 3)

    print(f'Fold: {f} {METRIC_NAME}: {score}')

    oofs[val_idx] = vp
    preds += tp/N_FOLDS

    fi_df = fi_df.append(pd.DataFrame({'ft': fts, 'imp': clf.feature_importances_}))

oof_score = round(rmse(train[TARGET_COL], oofs), 3)
print(f'\nOOF {METRIC_NAME}: {oof_score}')

cb_preds = preds.copy()
cb_oofs = oofs.copy()

Fold: 0 rmse: 1016.483
Fold: 1 rmse: 720.03
Fold: 2 rmse: 715.268
Fold: 3 rmse: 839.191
Fold: 4 rmse: 715.289
Fold: 5 rmse: 892.174
Fold: 6 rmse: 747.586
Fold: 7 rmse: 726.758
Fold: 8 rmse: 724.05
Fold: 9 rmse: 662.922
Fold: 10 rmse: 1001.246
Fold: 11 rmse: 751.781
Fold: 12 rmse: 699.415
Fold: 13 rmse: 812.252
Fold: 14 rmse: 765.726
Fold: 15 rmse: 1377.409
Fold: 16 rmse: 719.577
Fold: 17 rmse: 752.181
Fold: 18 rmse: 704.106
Fold: 19 rmse: 709.692
Fold: 20 rmse: 877.296
Fold: 21 rmse: 676.201
Fold: 22 rmse: 745.95
Fold: 23 rmse: 692.502
Fold: 24 rmse: 821.813

OOF rmse: 808.638


In [37]:
N_FOLDS = 15
folds = GroupKFold(N_FOLDS)
oofs, preds = np.zeros(len(train)), np.zeros(len(test))
fi_df = pd.DataFrame()

for f, (trn_idx, val_idx) in enumerate(folds.split(train, train[TARGET_COL], train[ID_COL])):

    X_trn, y_trn = train.iloc[trn_idx][fts], train[TARGET_COL].iloc[trn_idx]
    X_val, y_val = train.iloc[val_idx][fts], train[TARGET_COL].iloc[val_idx]

    clf = XGBRegressor(n_estimators=10000, learning_rate = 0.05, colsample_bytree=0.3, max_depth=5, random_state=2, seed=2)

    clf.fit(X_trn, y_trn, eval_set=[(X_val, y_val)], verbose=0, early_stopping_rounds=150)

    vp = clf.predict(X_val)
    tp = clf.predict(test[fts])

    score = round(rmse(y_val, vp), 3)

    print(f'Fold: {f} {METRIC_NAME}: {score}')

    oofs[val_idx] = vp
    preds += tp/N_FOLDS

    fi_df = fi_df.append(pd.DataFrame({'ft': fts, 'imp': clf.feature_importances_}))

oof_score = round(rmse(train[TARGET_COL], oofs), 3)
print(f'\nOOF {METRIC_NAME}: {oof_score}')

xgb_preds = preds.copy()
xgb_oofs = oofs.copy()

[13:09:54] WARNING: /workspace/src/objective/regression_obj.cu:152: reg:linear is now deprecated in favor of reg:squarederror.
Fold: 0 rmse: 848.953
[13:10:02] WARNING: /workspace/src/objective/regression_obj.cu:152: reg:linear is now deprecated in favor of reg:squarederror.
Fold: 1 rmse: 736.823
[13:10:26] WARNING: /workspace/src/objective/regression_obj.cu:152: reg:linear is now deprecated in favor of reg:squarederror.
Fold: 2 rmse: 698.533
[13:10:41] WARNING: /workspace/src/objective/regression_obj.cu:152: reg:linear is now deprecated in favor of reg:squarederror.
Fold: 3 rmse: 787.531
[13:10:49] WARNING: /workspace/src/objective/regression_obj.cu:152: reg:linear is now deprecated in favor of reg:squarederror.
Fold: 4 rmse: 745.254
[13:10:57] WARNING: /workspace/src/objective/regression_obj.cu:152: reg:linear is now deprecated in favor of reg:squarederror.
Fold: 5 rmse: 1222.182
[13:11:03] WARNING: /workspace/src/objective/regression_obj.cu:152: reg:linear is now deprecated in favor

In [38]:
#912, 809

In [39]:
sub = pd.DataFrame()
sub[ID_COL] = test[ID_COL].astype('str') + ' x ' + test['target_month'].astype('str')
sub['Target'] = (lgb_preds * 0.8 + cb_preds * 0.2) * 0.85 + xgb_preds * 0.15

In [40]:
ss["Target"] = ss[ID_COL].map(sub.set_index(ID_COL)["Target"])
ss.head()

,ID,Target
0,ID_ECQME2K x m1,1158.658672
1,ID_ECQME2K x m2,1156.675671
2,ID_ECQME2K x m3,1158.844773
3,ID_ECQME2K x m4,1157.678846
4,ID_ECQME2K x m5,1161.742751


In [51]:
ss.to_csv('paygo_lgb_cb_v3.csv', index=False)

In [52]:
oofs_df = train[[ID_COL, 'target_month', TARGET_COL]]
oofs_df['lgb'] = (lgb_oofs * 0.8 + cb_oofs * 0.2)*0.85 + xgb_oofs * 0.15
oofs_df['ID'] = oofs_df['ID'].astype('str') + ' x ' + oofs_df['target_month'].astype('str')

In [53]:
lstm_cnn = pd.read_csv('paygo_lstm_cnn_v6_oofs.csv')
oofs_df['lstm_cnn'] = oofs_df['ID'].map(lstm_cnn.set_index('ID')['Target'])

In [56]:
blnd = oofs_df['lgb'] * 0.8 + oofs_df['lstm_cnn'] * 0.2
rmse(oofs_df[TARGET_COL], blnd)

803.1962757214974

In [57]:
lgbcb_v3 = pd.read_csv('paygo_lgb_cb_v3.csv')
lstm_cnn_v6 = pd.read_csv('paygo_lstm_cnn_v6.csv')

In [58]:
np.corrcoef(lgbcb_v3['Target'], lstm_cnn_v6["Target"])

array([[1.        , 0.96487615],
       [0.96487615, 1.        ]])

In [59]:
blnd_sub = lgbcb_v3.copy()
blnd_sub['Target'] = (lgbcb_v3['Target'] * 0.8 + lstm_cnn_v6['Target'] * 0.2)
blnd_sub.to_csv('blend_sub_b.csv', index=False)